# Gemma calorie fine-tuning

This Colab notebook reads meal examples from Google Drive, builds a multimodal training dataset, and fine-tunes a Gemma E2B checkpoint with LoRA adapters.

Before running:
- set `DRIVE_DATASET_DIR`
- set `BASE_MODEL`
- make sure you have access to the target Gemma checkpoint


In [ ]:
!pip install -q transformers datasets peft trl accelerate bitsandbytes pillow pandas

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE_MODEL = 'SET_THIS_TO_YOUR_GEMMA_4_E2B_MODEL_ID'
DRIVE_DATASET_DIR = '/content/drive/MyDrive/calorie-training-data'
OUTPUT_DIR = '/content/drive/MyDrive/calorie-model-outputs/gemma-e2b-calorie-lora'
MAX_SAMPLES = None
IMAGE_TOKEN_TEXT = 'Estimate calories for this meal photo.'


In [ ]:
from pathlib import Path
import json
import pandas as pd

dataset_dir = Path(DRIVE_DATASET_DIR)
records = []

for metadata_path in sorted(dataset_dir.glob('*.json')):
    payload = json.loads(metadata_path.read_text())
    photo_name = payload.get('photo', {}).get('name')
    if not photo_name:
        continue
    image_path = dataset_dir / photo_name
    if not image_path.exists():
        continue
    records.append({
        'image_path': str(image_path),
        'description': payload['description'],
        'calorie_estimate': int(payload['calorie_estimate']),
        'captured_at': payload.get('captured_at'),
        'confidence_state': payload.get('confidence_state'),
    })

if MAX_SAMPLES is not None:
    records = records[:MAX_SAMPLES]

df = pd.DataFrame(records)
df.head(), len(df)

In [ ]:
from datasets import Dataset
from PIL import Image

dataset = Dataset.from_pandas(df)

def build_example(example):
    prompt = (
        f"{IMAGE_TOKEN_TEXT} User meal description: {example['description']}. "
        f"Return only the calorie estimate as a number."
    )
    answer = str(example['calorie_estimate'])
    return {
        'image': Image.open(example['image_path']).convert('RGB'),
        'prompt': prompt,
        'answer': answer,
    }

prepared = dataset.map(build_example, remove_columns=dataset.column_names)
prepared[0]

In [ ]:
import torch
from peft import LoraConfig, get_peft_model
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig, TrainingArguments
from trl import SFTTrainer

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

processor = AutoProcessor.from_pretrained(BASE_MODEL)
model = AutoModelForImageTextToText.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    torch_dtype=torch.bfloat16,
    device_map='auto',
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    task_type='CAUSAL_LM',
)

model = get_peft_model(model, lora_config)

def format_example(example):
    messages = [{
        'role': 'user',
        'content': [
            {'type': 'image', 'image': example['image']},
            {'type': 'text', 'text': example['prompt']},
        ],
    }, {
        'role': 'assistant',
        'content': [{'type': 'text', 'text': example['answer']}],
    }]
    return {'messages': messages}

train_dataset = prepared.map(format_example, remove_columns=prepared.column_names)

training_args = TrainingArguments(
    output_dir='/content/gemma-calorie-output',
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    num_train_epochs=2,
    logging_steps=10,
    save_strategy='epoch',
    bf16=True,
    report_to='none',
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    processing_class=processor,
)

trainer.train()

In [ ]:
from pathlib import Path

output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)
trainer.save_model(str(output_dir))
processor.save_pretrained(str(output_dir))
df.to_json(output_dir / 'training_manifest.jsonl', orient='records', lines=True)
print(f'Saved adapters and manifest to {output_dir}')